# Ridge Tier-1 Optimization: standardization + α CV + nonlinear transforms + rate acceleration

**Intent:** tighten the Ridge baseline before attempting finite-pool features (tier 2). Add the standard hygiene path_b_lite's original Ridge skipped — standardization, per-snap α CV, nonlinear transforms of the dominant feature, and a rate-acceleration feature.

**Reads from:** `notebooks/.cache/phase1_three_way.pkl` (library / ship / original-Ridge predictions + features).

**Changes vs. original Ridge (from `phase1_three_way.ipynb`):**
1. **Standardize** features via `StandardScaler` inside a `Pipeline`. Training-set stats only (no LOO leakage).
2. **CV α per snap** over {0.01, 0.1, 1, 10, 100, 1000}. Path_b_lite's α=10 was T-3d-tuned; other snaps probably differ.
3. **Nonlinear transforms:** `log(1 + rate_last_day)`, `sqrt(rate_last_day)`, `log(1 + observed_count)`.
4. **Rate acceleration:** `rate_delta = rate_last_day − rate_first_day`.

**Features: 10 base + 4 new = 14 total.** LOO fit per-snap with pipeline `StandardScaler → Ridge(α*)`.

**Plan doc:** `brainstorm/brainstorm_ridge_optimization.md`.


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold

import _helpers as H

print(f'cohort: {len(H.close_date_map)} movies')


## Load three-way cache

The `phase1_three_way.pkl` cache has per-(target, snap) rows with library_pred, ship_pred, original ridge_pred, plus the 10 base features. We'll add new features in-memory and fit tier-1 Ridge here.


In [ ]:
THREE_WAY_CACHE = H.CACHE_DIR / 'phase1_three_way.pkl'
if not THREE_WAY_CACHE.exists():
    raise FileNotFoundError(f'run phase1_three_way.ipynb first to build {THREE_WAY_CACHE}')

with open(THREE_WAY_CACHE, 'rb') as f:
    df = pickle.load(f)

print(f'loaded {len(df)} rows from {THREE_WAY_CACHE.name}')
print(f'snaps: {sorted(df["snap_days"].unique())}')
print(f'columns: {list(df.columns)}')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
BASE_FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]
NEW_FEATURES = [
    'log_observed_count', 'log_rate_last_day', 'sqrt_rate_last_day', 'rate_delta',
]
ALL_FEATURES = BASE_FEATURES + NEW_FEATURES

ALPHA_GRID = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
CV_FOLDS = 5
CV_SEED = 42

print(f'{len(ALL_FEATURES)} features, {len(ALPHA_GRID)} α candidates, {CV_FOLDS}-fold CV')


## Add new features

Computed directly from existing columns — no re-fetching from the reviews DataFrame.


In [ ]:
df['log_observed_count'] = np.log1p(df['observed_count'])
df['log_rate_last_day'] = np.log1p(df['rate_last_day'])
df['sqrt_rate_last_day'] = np.sqrt(df['rate_last_day'].clip(lower=0))
df['rate_delta'] = df['rate_last_day'] - df['rate_first_day']

# Sanity check
print('new feature summary:')
print(df[NEW_FEATURES].describe().round(2).to_string())
print()
print(f'null counts in ALL_FEATURES: {df[ALL_FEATURES].isna().sum().sum()}')


## Per-snap α CV selection

For each snap, 5-fold CV across all valid targets over the α grid. Pick argmin-MAE α. This chooses one α per snap (not per LOO iter), which is a standard convention and adds at most mild optimism.


In [ ]:
def select_alpha_by_cv(X, y, alpha_grid, folds=CV_FOLDS, seed=CV_SEED):
    kf = KFold(n_splits=folds, shuffle=True, random_state=seed)
    best_alpha = None
    best_mae = np.inf
    alpha_maes = {}
    for alpha in alpha_grid:
        fold_errs = []
        for train_idx, test_idx in kf.split(X):
            pipe = Pipeline([
                ('scaler', StandardScaler()),
                ('ridge', Ridge(alpha=alpha)),
            ])
            pipe.fit(X[train_idx], y[train_idx])
            preds = pipe.predict(X[test_idx])
            fold_errs.extend(np.abs(preds - y[test_idx]).tolist())
        mae = float(np.mean(fold_errs))
        alpha_maes[alpha] = mae
        if mae < best_mae:
            best_mae = mae
            best_alpha = alpha
    return best_alpha, alpha_maes


snap_alpha = {}
print('α selection per snap:')
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < CV_FOLDS * 2:
        print(f'  T-{snap_days}d: skipped (n={len(sub)} too small)')
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    best_alpha, alpha_maes = select_alpha_by_cv(X, y, ALPHA_GRID)
    snap_alpha[snap_days] = best_alpha
    alpha_str = '  '.join(f'{a}={alpha_maes[a]:.2f}' for a in ALPHA_GRID)
    print(f'  T-{snap_days}d (n={len(sub)}): α*={best_alpha}   {alpha_str}')


## LOO fit+predict with best α per snap

Pipeline `StandardScaler → Ridge(α*)` refit per LOO iter (training-set stats only, no leakage).


In [ ]:
df['ridge_t1_pred'] = np.nan
for snap_days in SNAP_DAYS_LIST:
    alpha = snap_alpha.get(snap_days)
    if alpha is None:
        continue
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual']).copy()
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    idx = sub.index.values

    preds = np.zeros(len(sub))
    for i in range(len(sub)):
        mask = np.ones(len(sub), dtype=bool)
        mask[i] = False
        pipe = Pipeline([
            ('scaler', StandardScaler()),
            ('ridge', Ridge(alpha=alpha)),
        ])
        pipe.fit(X[mask], y[mask])
        preds[i] = pipe.predict(X[i:i+1])[0]
    df.loc[idx, 'ridge_t1_pred'] = preds
    print(f'  T-{snap_days}d: LOO fit {len(sub)} models at α={alpha}')

CACHE = H.CACHE_DIR / 'phase1_ridge_tier1.pkl'
with open(CACHE, 'wb') as f:
    pickle.dump(df, f)
print(f'saved {CACHE}')


## Per-variant summary (4-way)


In [ ]:
def variant_metrics(sub, pred_col):
    s = sub.dropna(subset=[pred_col]).copy()
    if len(s) == 0:
        return None
    err = s[pred_col].values - s['actual'].values
    abs_err = np.abs(err)
    pred = s[pred_col].values
    actual = s['actual'].values.astype(float)
    safe_actual = np.where(actual > 0, actual, np.nan)
    return {
        'n': len(s), 'MAE': float(abs_err.mean()),
        'me': float(err.mean()),
        'med_err': float(np.median(err)),
        'med_abs_err': float(np.median(abs_err)),
        'p90_abs_err': float(np.quantile(abs_err, 0.9)),
        'med_ratio': float(np.nanmedian(pred / safe_actual)),
    }


PRED_COLS = [
    ('library', 'lib_pred'),
    ('ship', 'ship_pred'),
    ('ridge_orig', 'ridge_pred'),
    ('ridge_t1', 'ridge_t1_pred'),
]

summary_rows = []
print('=== per-variant summary ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    print(f'T-{snap_days}d')
    for name, col in PRED_COLS:
        m = variant_metrics(sub, col)
        if m is None:
            continue
        summary_rows.append({'snap_days': snap_days, 'variant': name, **m})
        print(f'  {name:12s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  '
              f'me={m["me"]:+6.2f}  med_err={m["med_err"]:+6.2f}  '
              f'med|e|={m["med_abs_err"]:6.2f}  p90|e|={m["p90_abs_err"]:6.2f}  '
              f'med_ratio={m["med_ratio"]:5.2f}')
    print()
summary = pd.DataFrame(summary_rows)


## Δ vs ship and vs ridge_orig


In [ ]:
print(f'{"snap":<6}{"variant":<12}{"MAE":>8}{"Δ vs ship":>12}{"% vs ship":>12}'
      f'{"Δ vs orig":>12}{"% vs orig":>12}')
for snap_days in SNAP_DAYS_LIST:
    ship_row = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == 'ship')]
    orig_row = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == 'ridge_orig')]
    if ship_row.empty or orig_row.empty:
        continue
    ship_mae = ship_row.iloc[0]['MAE']
    orig_mae = orig_row.iloc[0]['MAE']
    for name, col in PRED_COLS:
        m = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == name)]
        if m.empty:
            continue
        mae = m.iloc[0]['MAE']
        d_ship = ship_mae - mae
        d_orig = orig_mae - mae
        p_ship = 100 * d_ship / ship_mae if ship_mae else float('nan')
        p_orig = 100 * d_orig / orig_mae if orig_mae else float('nan')
        print(f'T-{snap_days}d  {name:<12}{mae:>8.2f}{d_ship:>+12.2f}{p_ship:>+11.2f}%'
              f'{d_orig:>+12.2f}{p_orig:>+11.2f}%')
    print()


## Paired bootstrap: ridge_t1 vs {ship, ridge_orig}

Per target: delta = |err_A| − |err_B|. Positive = B wins.


In [ ]:
PAIRS = [('ship', 'ridge_t1'), ('ridge_orig', 'ridge_t1')]
COL_MAP = {'library': 'lib_pred', 'ship': 'ship_pred',
           'ridge_orig': 'ridge_pred', 'ridge_t1': 'ridge_t1_pred'}

print('=== paired bootstrap ΔMAE (A − B, +ve → B wins) ===\n')
print(f'{"snap":<6}{"A":<12}{"B":<12}{"Δ units":>10}{"CI95_lo":>10}{"CI95_hi":>10}'
      f'{"Δ %":>10}{"n":>6}  result')

for snap_days in SNAP_DAYS_LIST:
    snap_df = df[df['snap_days'] == snap_days]
    for a_name, b_name in PAIRS:
        a_col, b_col = COL_MAP[a_name], COL_MAP[b_name]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual'])
        if len(paired) < 5:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        deltas = a_abs - b_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        a_mae = a_abs.mean()
        pct = 100 * point / a_mae if a_mae else float('nan')
        if lo > 0:
            result = f'{b_name:>10} wins'
        elif hi < 0:
            result = f'{a_name:>10} wins'
        else:
            result = '    ns'
        print(f'T-{snap_days}d  {a_name:<12}{b_name:<12}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}'
              f'{pct:>+10.2f}{len(paired):>6}  {result}')
    print()


## h/m subset breakdown


In [ ]:
HM = ['the_drama', 'the_super_mario_galaxy_movie', 'forbidden_fruits_2026',
      'they_will_kill_you', 'you_me_and_tuscany']
hm_df = df[df['target_slug'].isin(HM)]

print('=== h/m subset per snap (4-way) ===\n')
for snap_days in SNAP_DAYS_LIST:
    sub = hm_df[hm_df['snap_days'] == snap_days]
    if sub.empty:
        continue
    print(f'T-{snap_days}d  (n={len(sub)} h/m targets)')
    for name, col in PRED_COLS:
        s = sub.dropna(subset=[col])
        if len(s) == 0:
            continue
        err = s[col].values - s['actual'].values
        abs_err = np.abs(err)
        print(f'  {name:12s}  n={len(s)}  MAE={abs_err.mean():7.2f}  me={err.mean():+7.2f}')
    print()


## Feature coefficients at each snap (interpretability)


In [ ]:
print('=== learned Ridge coefficients (standardized scale) ===\n')
for snap_days in SNAP_DAYS_LIST:
    alpha = snap_alpha.get(snap_days)
    if alpha is None:
        continue
    sub = df[df['snap_days'] == snap_days].dropna(subset=ALL_FEATURES + ['actual'])
    if len(sub) < 10:
        continue
    X = sub[ALL_FEATURES].values
    y = sub['actual'].values.astype(float)
    pipe = Pipeline([('scaler', StandardScaler()), ('ridge', Ridge(alpha=alpha))])
    pipe.fit(X, y)
    coefs = pipe.named_steps['ridge'].coef_
    intercept = pipe.named_steps['ridge'].intercept_
    pairs = sorted(zip(ALL_FEATURES, coefs), key=lambda p: -abs(p[1]))
    print(f'T-{snap_days}d  α={alpha}  intercept={intercept:.2f}')
    for feat, c in pairs:
        print(f'  {feat:28s}  {c:+7.2f}')
    print()


## Plot: MAE by snap, all 4 variants


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
colors = {'library': 'tab:gray', 'ship': 'tab:red',
          'ridge_orig': 'tab:blue', 'ridge_t1': 'tab:green'}
markers = {'library': 's', 'ship': 'o', 'ridge_orig': '^', 'ridge_t1': 'D'}
for name, _ in PRED_COLS:
    sub = summary[summary['variant'] == name].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['MAE'], '-',
            marker=markers[name], color=colors[name], label=name, markersize=8)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('MAE (reviews)')
ax.set_title('Phase-1 MAE — tier-1 Ridge vs baselines')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
